# 01 — Landmark Extraction

Extract **33 MediaPipe landmarks** from exercise videos using the Heavy model.

## What this does
- Reads `.mov` or `.mp4` videos from `data/raw_videos/{exercise}/`
- Runs MediaPipe PoseLandmarker Heavy on every frame (Tasks API)
- Checks if each frame has a usable pose detection (major joints visible)
- Saves per-video: `.npz` (landmarks + visibility arrays) and `.json` (metadata)

## Inputs
```
data/raw_videos/pushup/pushup.mov   (or .mp4)
data/raw_videos/lunge/lunge.mov     (or .mp4)
```

## Outputs
```
data/extracted/pushup.npz   — landmarks (N, 33, 3), visibility (N, 33)
data/extracted/pushup.json  — metadata (exercise, fps, usable frame count)
```

---

**Phase 1 focus:** Start with `pushup.mov`, then add `lunge.mov`.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.landmark_extractor import extract_all, extract_video, save_results
from src.video_utils import convert_all_mov
from pathlib import Path
import numpy as np
import json
import shutil

## Step 1: Set up data directories

Copy source videos from `research/data/portrait/` into the pipeline's expected location.
The extractor looks for videos in `data/raw_videos/{exercise}/`.

In [ ]:
# Source video locations
SOURCE_DIR = Path('../research/data/portrait')
RAW_DIR = Path('../data/raw_videos')

# Create exercise folders and copy videos
for exercise, filename in [('pushup', 'pushup.mov'), ('lunge', 'lunge.mov')]:
    src = SOURCE_DIR / filename
    dst_dir = RAW_DIR / exercise
    dst = dst_dir / filename
    
    if src.exists() and not dst.exists():
        dst_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print(f'Copied: {src} -> {dst}')
    elif dst.exists():
        print(f'Already exists: {dst}')
    else:
        print(f'Source not found: {src}')

# Show what we have
print(f'\nVideos ready for extraction:')
for exercise_dir in sorted(RAW_DIR.iterdir()):
    if exercise_dir.is_dir():
        videos = list(exercise_dir.glob('*.*'))
        print(f'  {exercise_dir.name}/: {[v.name for v in videos]}')

## Step 2: (Optional) Convert .mov to .mp4

OpenCV reads `.mov` files natively, so **skip this** unless you specifically need `.mp4`.
If you do convert, the extractor will automatically de-duplicate and only process one version per video.

In [ ]:
# Optional: convert .mov files to .mp4 (requires ffmpeg)
# Uncomment the line below to run conversion:
convert_all_mov(str(RAW_DIR))

## Step 3: Extract landmarks from all videos

This runs MediaPipe Pose Heavy on every frame. Takes a few minutes per video.

In [ ]:
results = extract_all(
    raw_videos_dir=str(RAW_DIR),
    output_dir='../data/extracted',
)

# Summary
print(f'\n{"="*60}')
print('EXTRACTION SUMMARY')
print(f'{"="*60}')
for r in results:
    pct = 100 * r['usable_frames'] / max(r['total_frames'], 1)
    print(f"  {r['exercise']:>8s} | {r['video']:>20s} | {r['usable_frames']:>5d}/{r['total_frames']:<5d} usable ({pct:.1f}%)")

## Step 4: Quick quality check

Load one extracted file and verify shapes and quality.

In [ ]:
extracted_dir = Path('../data/extracted')
npz_files = sorted(extracted_dir.glob('*.npz'))

if npz_files:
    for npz_file in npz_files:
        sample = np.load(npz_file)
        json_file = npz_file.with_suffix('.json')
        
        print(f'--- {npz_file.name} ---')
        print(f'  Landmarks shape: {sample["landmarks"].shape}')  # (N, 33, 3)
        print(f'  Visibility shape: {sample["visibility"].shape}')  # (N, 33)
        print(f'  Mean visibility: {sample["visibility"].mean():.3f}')
        
        if json_file.exists():
            with open(json_file) as f:
                meta = json.load(f)
            print(f'  Exercise: {meta["exercise"]}')
            print(f'  Usable: {meta["usable_frames"]}/{meta["extracted_frames"]} ({meta["usable_pct"]}%)')
            print(f'  Avg visibility: {meta["avg_visibility"]}')
        print()
else:
    print('No extracted files found. Run extraction first.')

---
**Next:** Run `02_visualize_validate.ipynb` to visually inspect the skeleton overlays.